# CDS 4003 – Decision Tree (Python)

This notebook evaluates **Pass / Fail** predictions using the prepared `CleanedDataCDS4003.csv` dataset and **10-fold stratified cross-validation**.

**Important:** This is a Python reimplementation of the Decision Tree model workflow performed in **Altair AI Studio**. The Python implementation uses a different algorithm configuration (Entropy instead of Gain Ratio), so the results may not match exactly. Keep the actual Altair results in the Submission 3 report.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# 2. Load the cleaned dataset
df = pd.read_csv("CleanedDataCDS4003.csv", sep=";")

print("Dataset shape:", df.shape)
print("Pass/Fail counts:")
print(df["pass_status"].value_counts())

In [ ]:
# 3. Define features and target
features = [
    "time_management_score",
    "previous_gpa",
    "attendance_percentage",
    "previous_exam_score",
    "assignment_completion_rate",
    "study_hours_per_day",
    "self_study_hours",
    "practice_tests_completed",
    "exam_preparation_days"
]

X = df[features]
y = df["pass_status"].str.strip().str.title()

assert not X.isna().any().any()
assert set(y.unique()) == {"Pass", "Fail"}

In [ ]:
# 4. Build Decision Tree
model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

In [ ]:
# 5. Apply 10-fold cross-validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=2001
)

results = []
predictions = np.empty(len(y), dtype=object)

for fold, (train_index, test_index) in enumerate(
    cv.split(X, y), start=1
):
    model.fit(X.iloc[train_index], y.iloc[train_index])

    pred = model.predict(X.iloc[test_index])
    predictions[test_index] = pred

    actual = y.iloc[test_index]

    results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(actual, pred),
        "FAIL Precision": precision_score(
            actual, pred, pos_label="Fail", zero_division=0
        ),
        "FAIL Recall": recall_score(
            actual, pred, pos_label="Fail", zero_division=0
        ),
        "FAIL F1": f1_score(
            actual, pred, pos_label="Fail", zero_division=0
        )
    })

results_df = pd.DataFrame(results)

print("10-Fold Cross-Validation Results (Python):")
display(results_df.round(4))

print("\nMean scores:")
print(
    results_df.drop(columns="Fold")
    .mean()
    .map(lambda v: f"{v:.2%}")
)

print(
    "\nAccuracy variation (standard deviation):",
    f"{results_df['Accuracy'].std():.2%}"
)

In [ ]:
# 6. Confusion Matrix
labels = ["Fail", "Pass"]

cm = confusion_matrix(y, predictions, labels=labels)

print(pd.DataFrame(
    cm,
    index=["Actual Fail", "Actual Pass"],
    columns=["Predicted Fail", "Predicted Pass"]
))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
).plot(cmap="Blues", values_format="d", colorbar=False)

plt.title("Decision Tree – 10-Fold CV (Python)")
plt.show()

## How to interpret

- **Accuracy**: The overall percentage of students correctly classified as Pass or Fail.
- **FAIL Precision**: Among students predicted to fail, how many actually failed.
- **FAIL Recall**: Among students who actually failed, how many the model successfully identified.
- **FAIL F1**: Balances FAIL precision and recall to measure the model's performance.
- **Cross-validation**: Trains and tests the model using ten different splits to evaluate its performance and consistency.

**Submission 3 values from Altair (keep separate):** Accuracy **79.41% ± 0.29%**, FAIL Recall **22.11% ± 0.96%**, and FAIL Precision **62.65% ± 1.77%**. Python results may differ.

**Important limitation:** The dataset was already normalized during the earlier Altair preparation, before cross-validation. For a more reliable evaluation, preprocessing should be performed within each training fold to avoid data leakage.